In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ==============================================================================
# 步骤 1: 格式化磁盘 & 安装环境
# ==============================================================================
import os

# --- 1. 核弹级清理 (释放 17.1GB) ---
print("🧹 正在清空 /kaggle/working 以释放空间...")
%cd /kaggle/working
!rm -rf * # 删除当前目录下所有文件
!rm -rf .git
!rm -rf /root/.cache/pip
!pip cache purge > /dev/null
print("✅ 磁盘清理完成！现在你有完整的 20GB 空间了。")

# --- 2. 安装 ComfyUI ---
print("\n📦 正在安装 ComfyUI...")
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install -r requirements.txt > /dev/null
# 锁定依赖版本，防止报错
!pip install -U "huggingface_hub<1.0" "pydantic<2.12" accelerate diffusers gguf bitsandbytes pyngrok > /dev/null

# --- 3. 安装关键插件 ---
print("\n🔌 正在安装插件 (WanWrapper + GGUF)...")
# Wan 封装插件
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper custom_nodes/ComfyUI-WanVideoWrapper
!pip install -r custom_nodes/ComfyUI-WanVideoWrapper/requirements.txt > /dev/null

# GGUF 解码插件 (必装)
!git clone https://github.com/city96/ComfyUI-GGUF custom_nodes/ComfyUI-GGUF
!pip install -r custom_nodes/ComfyUI-GGUF/requirements.txt > /dev/null

print("✨ 环境安装完毕！")

In [6]:
# ==============================================================================
# 步骤 2: 挂载 Input 模型 + 补全 VAE/CLIP
# ==============================================================================
import os
import glob
from huggingface_hub import hf_hub_download

# 定义目录
base_dir = "/kaggle/working/ComfyUI/models"
unet_dir = f"{base_dir}/unet"
vae_dir = f"{base_dir}/vae"
clip_dir = f"{base_dir}/clip"
os.makedirs(unet_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)

# --- 1. 挂载主模型 (从 Input 目录找) ---
print("🔍 正在扫描 Input 数据集...")
# 扫描所有 .gguf 文件
gguf_files = glob.glob("/kaggle/input/**/*.gguf", recursive=True)

if gguf_files:
    for model_path in gguf_files:
        filename = os.path.basename(model_path)
        # 创建软链接 (快捷方式)
        target = os.path.join(unet_dir, filename)
        if not os.path.exists(target):
            os.symlink(model_path, target)
            print(f"✅ 已挂载主模型: {filename}")
else:
    print("❌ 警告：在 Input 里没找到 GGUF 模型！请确认你是否 Add Input 了数据集。")

# --- 2. 补全 VAE & CLIP (如果 Dataset 里没有，就自动下载) ---
print("\n🔍 检查 VAE 和 CLIP...")

# 检查 VAE
vae_path = f"{vae_dir}/wan2.2_vae.safetensors"
if not os.path.exists(vae_path):
    # 尝试去 Input 里找
    input_vaes = glob.glob("/kaggle/input/**/*vae*.safetensors", recursive=True)
    if input_vaes:
        os.symlink(input_vaes[0], vae_path)
        print("✅ 已挂载 VAE (来自 Input)")
    else:
        print("⬇️ 正在下载 VAE (Input里没有)...")
        hf_hub_download(repo_id="Comfy-Org/Wan_2.2_ComfyUI_Repackaged", filename="split_files/vae/wan2.2_vae.safetensors", local_dir=vae_dir, local_dir_use_symlinks=False)
        # 移动文件 (因为下载会带子目录)
        if os.path.exists(f"{vae_dir}/split_files/vae/wan2.2_vae.safetensors"):
             os.rename(f"{vae_dir}/split_files/vae/wan2.2_vae.safetensors", vae_path)
             import shutil; shutil.rmtree(f"{vae_dir}/split_files")

# 检查 CLIP (T5)
clip_path = f"{clip_dir}/umt5_xxl_fp8_e4m3fn_scaled.safetensors"
if not os.path.exists(clip_path):
    # 尝试去 Input 里找
    input_clips = glob.glob("/kaggle/input/**/*umt5*.safetensors", recursive=True)
    if input_clips:
        os.symlink(input_clips[0], clip_path)
        print("✅ 已挂载 CLIP (来自 Input)")
    else:
        print("⬇️ 正在下载 T5 CLIP (Input里没有)...")
        hf_hub_download(repo_id="Comfy-Org/Wan_2.2_ComfyUI_Repackaged", filename="split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", local_dir=clip_dir, local_dir_use_symlinks=False)
        if os.path.exists(f"{clip_dir}/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"):
             os.rename(f"{clip_dir}/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", clip_path)
             import shutil; shutil.rmtree(f"{clip_dir}/split_files")

print("\n✨ 模型准备就绪！")
!ls -lh models/unet models/vae models/clip

🔍 正在扫描 Input 数据集...
✅ 已挂载主模型: wan2.1-t2v-14b-Q4_K_M.gguf

🔍 检查 VAE 和 CLIP...
✅ 已挂载 VAE (来自 Input)
✅ 已挂载 CLIP (来自 Input)

✨ 模型准备就绪！
models/clip:
total 4.0K
-rw-r--r-- 1 root root   0 Feb 11 05:42 put_clip_or_text_encoder_models_here
lrwxrwxrwx 1 root root 102 Feb 11 05:43 umt5_xxl_fp8_e4m3fn_scaled.safetensors -> /kaggle/input/datasets/jarredstroman/wan2-1/ComfyUI/models/clip/umt5_xxl_fp8_e4m3fn_scaled.safetensors

models/unet:
total 4.0K
-rw-r--r-- 1 root root  0 Feb 11 05:42 put_unet_files_here
lrwxrwxrwx 1 root root 90 Feb 11 05:43 wan2.1-t2v-14b-Q4_K_M.gguf -> /kaggle/input/datasets/jarredstroman/wan2-1/ComfyUI/models/unet/wan2.1-t2v-14b-Q4_K_M.gguf

models/vae:
total 4.0K
-rw-r--r-- 1 root root  0 Feb 11 05:42 put_vae_here
lrwxrwxrwx 1 root root 85 Feb 11 05:43 wan2.2_vae.safetensors -> /kaggle/input/datasets/jarredstroman/wan2-1/ComfyUI/models/vae/wan2.2_vae.safetensors


In [7]:
# 进入插件目录
%cd /kaggle/working/ComfyUI/custom_nodes

# 克隆 VideoHelperSuite 仓库
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git

# 安装该插件所需的 Python 依赖
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt

/kaggle/working/ComfyUI/custom_nodes
Cloning into 'ComfyUI-VideoHelperSuite'...
remote: Enumerating objects: 3332, done.
remote: Counting objects: 100% (1589/1589), done.
remote: Compressing objects: 100% (359/359), done.
remote: Total 3332 (delta 1454), reused 1234 (delta 1229), pack-reused 1743 (from 2)
Receiving objects: 100% (3332/3332), 789.83 KiB | 8.49 MiB/s, done.
Resolving deltas: 100% (1968/1968), done.


In [10]:
# ==============================================================================
# 步骤 3: 启动 ComfyUI
# ==============================================================================
import os
from pyngrok import ngrok

# 1. 强制开启双 T4 显卡
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# 2. Ngrok 配置 (已填入你提供的 Token)
NGROK_TOKEN = "394HSCYBOyi1ZXm5heUutThsefX_5W6E2JWnccTPNpmJauexK"
ngrok.set_auth_token(NGROK_TOKEN)

try:
    !pkill -f ngrok
    public_url = ngrok.connect(8188)
    print(f"🚀 ComfyUI 已启动！请点击访问: {public_url}")
except Exception as e:
    print(f"⚠️ Ngrok 启动提示: {e}")

# 3. 启动命令 (High VRAM 模式，适配 T4 双卡)
!python main.py --listen --port 8188 --highvram --reserve-vram 0.5

🚀 ComfyUI 已启动！请点击访问: NgrokTunnel: "https://carol-suppositionless-nonideologically.ngrok-free.dev" -> "http://localhost:8188"
python3: can't open file '/kaggle/working/ComfyUI/custom_nodes/main.py': [Errno 2] No such file or directory
